In [1]:
### INSTALLATION & TESTING BEFORE FULLY RUN ###

In [2]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
import time
import ollama
import pandas as pd
import numpy as np

from pathlib import Path
from collections import Counter
from datetime import datetime

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
BASE_DIR = Path.cwd()

DATASET_FILE = (
    BASE_DIR
    / "malay_cleaned_dataset"
    / "malay_evaluation_dataset.jsonl"
)

RESULT_DIR = BASE_DIR / "results" / "qwen"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_FILE = RESULT_DIR / "pilot_results.jsonl"
PREDICTION_FILE = RESULT_DIR / "predictions.jsonl"
CHECKPOINT_FILE = RESULT_DIR / "checkpoint.jsonl"
METRICS_FILE = RESULT_DIR / "metrics.csv"

print("Dataset exists:", DATASET_FILE.exists())
print("Result directory:", RESULT_DIR.resolve())

Dataset exists: True
Result directory: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\results\qwen


In [3]:
try:
    ollama_response = ollama.list()

    print("Ollama connection: SUCCESS")
    print("Ollama is ready!")

except Exception as e:
    print("Ollama connection: FAILED")
    print("Error:", e)

Ollama connection: SUCCESS
Ollama is ready!


In [4]:
records = []

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

print("Total records:", len(records))
print("\nDomain distribution:")

display(
    df["domain"]
    .value_counts()
    .to_frame("Records")
)

Total records: 3448

Domain distribution:


,Records
domain,
Sentimen News,980
Penukaran Kod,799
Terjemahan,712
Sentimen,682
Campuran Kod,275


In [5]:
MODEL_NAME = "qwen3.8:27b"

GENERATION_CONFIG = {
    "temperature": 0,
    "seed": 42,
    "num_predict": 1024
}

print("Model:", MODEL_NAME)
print("Configuration:", GENERATION_CONFIG)

Model: qwen3.8:27b
Configuration: {'temperature': 0, 'seed': 42, 'num_predict': 1024}


In [6]:
installed_models = [
    model["model"]
    for model in ollama.list()["models"]
]

if MODEL_NAME in installed_models:
    print("Qwen 3.8 27B: AVAILABLE")
else:
    raise RuntimeError(
        f"Model {MODEL_NAME} not found in Ollama!"
    )

Qwen 3.8 27B: AVAILABLE


In [9]:
# ==========================================
# LOAD FULL DATASET
# ==========================================

EXPECTED_RECORDS = 3448
records = []

with open(
    DATASET_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)

print("DATASET LOADED")
print("=" * 55)

print("Total records:", len(df))

print("\nDomain distribution:")
print(df["domain"].value_counts())

assert len(df) == EXPECTED_RECORDS, (
    f"Expected {EXPECTED_RECORDS}, got {len(df)}"
)

assert df["domain"].notna().all()

print("\nDataset validation passed.")


DATASET LOADED
Total records: 3448

Domain distribution:
domain
Sentimen News    980
Penukaran Kod    799
Terjemahan       712
Sentimen         682
Campuran Kod     275
Name: count, dtype: int64

Dataset validation passed.


In [10]:
# ==========================================
# PROMPT CONFIGURATION
# ==========================================

PROMPTS = {

"Campuran Kod": """
Anda ialah pembantu yang menjana respons
berdasarkan konteks perbualan harian.

TUGAS:
Anda akan diberikan satu ujaran, soalan atau
mesej daripada seseorang.

Hasilkan DUA respons yang sesuai untuk membalas
input tersebut:

1. FORMAL:
   Gunakan Bahasa Melayu formal, baku dan sopan.

2. TIDAK FORMAL:
   Gunakan Bahasa Melayu santai dan semula jadi
   seperti perbualan harian.

ARAHAN PENTING:
- Fahami maksud dan konteks input terlebih dahulu.
- Jawab soalan atau berikan respons yang sesuai
  terhadap mesej yang diberikan.
- Jangan mengolah semula ayat input.
- Jangan menterjemahkan input.
- Jangan mengulang input sebagai jawapan.
- Pastikan kedua-dua respons benar-benar menjawab
  input yang diberikan.
- Kedua-dua respons mesti menyampaikan maksud
  jawapan yang sama tetapi menggunakan gaya
  bahasa yang berbeza.
- Jangan tambah penerangan atau ulasan.

FORMAT OUTPUT:
FORMAL: [Jawapan formal]
TIDAK FORMAL: [Jawapan tidak formal]

INPUT:
{input}
""",

"Penukaran Kod": """
Anda ialah pembantu yang menjana respons
berdasarkan konteks komunikasi profesional.

TUGAS:
Anda akan diberikan satu ujaran, soalan atau
mesej daripada seseorang.

Hasilkan DUA respons yang sesuai untuk membalas
input tersebut:

1. FORMAL:
   Gunakan Bahasa Melayu yang formal, baku,
   profesional dan sopan.

2. TIDAK FORMAL:
   Gunakan gaya bahasa yang lebih santai,
   semula jadi dan fleksibel, tetapi masih
   sesuai dalam konteks komunikasi profesional.
   Penggunaan Bahasa Melayu dan Bahasa Inggeris
   secara bercampur dibenarkan jika sesuai.

ARAHAN PENTING:
- Fahami maksud dan konteks input terlebih dahulu.
- Jawab soalan atau berikan respons yang sesuai
  terhadap mesej yang diberikan.
- Jangan mengolah semula ayat input.
- Jangan menterjemahkan input.
- Jangan mengulang input sebagai jawapan.
- Pastikan kedua-dua respons menjawab input.
- Kedua-dua respons mesti menyampaikan maksud
  jawapan yang sama tetapi menggunakan gaya
  bahasa yang berbeza.
- Jangan tambah penerangan atau ulasan.

FORMAT OUTPUT:
FORMAL: [Jawapan formal]
TIDAK FORMAL: [Jawapan tidak formal]

INPUT:
{input}
""",

"Terjemahan": """
Anda ialah penterjemah profesional
Bahasa Melayu dan Bahasa Inggeris.

TUGAS:
Terjemahkan teks yang diberikan ke bahasa
yang berbeza daripada bahasa sumber.

ARAHAN PENTERJEMAHAN:
- Jika teks sumber dalam Bahasa Inggeris,
  terjemahkan kepada Bahasa Melayu.
- Jika teks sumber dalam Bahasa Melayu,
  terjemahkan kepada Bahasa Inggeris.

ARAHAN PENTING:
- Kenal pasti bahasa sumber sebelum menterjemah.
- Gunakan bahasa sasaran yang betul.
- Kekalkan maksud asal dan konteks ayat.
- Pastikan semua maklumat penting dikekalkan.
- Gunakan tatabahasa dan struktur ayat
  yang betul serta semula jadi.
- Terjemahkan berdasarkan maksud dan konteks,
  bukan secara literal semata-mata.
- Jangan tambah atau buang maklumat penting.
- Jangan berikan penerangan atau ulasan.
- Jangan ulang teks sumber.

FORMAT OUTPUT:
Berikan hasil terjemahan sahaja.

TEKS SUMBER:
{input}
""",

    "Sentimen": """
Anda ialah penganalisis sentimen bahasa Melayu.

Jawab soalan berdasarkan pilihan jawapan yang diberikan.

Berikan jawapan dalam bentuk label sahaja:
Positif, Neutral atau Negatif.

Soalan:
{input}
""",


"Sentimen News": """
Anda ialah penganalisis sentimen berita Bahasa Melayu.

TUGAS:
Analisis teks berita yang diberikan dan tentukan
sifat sentimennya.

PILIH SATU LABEL SAHAJA:
- positif
- neutral
- negatif

PANDUAN:
- Positif: Berita menyampaikan perkembangan,
  keadaan atau peristiwa yang bersifat positif.
- Negatif: Berita menyampaikan perkembangan,
  keadaan atau peristiwa yang bersifat negatif.
- Neutral: Berita bersifat informatif atau
  melaporkan fakta tanpa sentimen positif
  atau negatif yang jelas.

ARAHAN PENTING:
- Nilai sentimen berdasarkan keseluruhan teks berita.
- Jangan membuat andaian berdasarkan satu perkataan sahaja.
- Jangan berikan penerangan atau alasan.
- Jangan keluarkan lebih daripada satu label.

FORMAT OUTPUT:
Berikan satu label sahaja: positif, neutral atau negatif.

TEKS BERITA:
{input}
"""

}

In [11]:
import re

def clean_question_number(text):
    """
    Buang nombor soalan pada permulaan teks.
    Contoh: '26) You cannotlah...' -> 'You cannotlah...'
    """
    if not isinstance(text, str):
        return text

    return re.sub(r"^\s*\d+\)\s*", "", text).strip()

In [12]:
def build_prompt(record):
    domain = record["domain"]

    if domain not in PROMPTS:
        raise ValueError(f"Unknown domain: {domain}")

    input_values = record["input"]

    input_text = "\n".join(
        f"{key}: {value}"
        for key, value in input_values.items()
        if value is not None
    )

    # Preprocessing khusus untuk Campuran Kod
    if domain == "Campuran Kod":
        input_text = re.sub(
            r"(^|\n)(korpus ujaran online:\s*)\d+\)\s*",
            r"\1\2",
            input_text,
            flags=re.IGNORECASE
        )

    return PROMPTS[domain].format(input=input_text)

In [13]:
for domain in PROMPTS:

    sample = next(
        record for record in records
        if record["domain"] == domain
    )

    print("\n" + "=" * 70)
    print("DOMAIN:", domain)
    print("=" * 70)

    print(build_prompt(sample))


DOMAIN: Campuran Kod

Anda ialah pembantu yang menjana respons
berdasarkan konteks perbualan harian.

TUGAS:
Anda akan diberikan satu ujaran, soalan atau
mesej daripada seseorang.

Hasilkan DUA respons yang sesuai untuk membalas
input tersebut:

1. FORMAL:
   Gunakan Bahasa Melayu formal, baku dan sopan.

2. TIDAK FORMAL:
   Gunakan Bahasa Melayu santai dan semula jadi
   seperti perbualan harian.

ARAHAN PENTING:
- Fahami maksud dan konteks input terlebih dahulu.
- Jawab soalan atau berikan respons yang sesuai
  terhadap mesej yang diberikan.
- Jangan mengolah semula ayat input.
- Jangan menterjemahkan input.
- Jangan mengulang input sebagai jawapan.
- Pastikan kedua-dua respons benar-benar menjawab
  input yang diberikan.
- Kedua-dua respons mesti menyampaikan maksud
  jawapan yang sama tetapi menggunakan gaya
  bahasa yang berbeza.
- Jangan tambah penerangan atau ulasan.

FORMAT OUTPUT:
FORMAL: [Jawapan formal]
TIDAK FORMAL: [Jawapan tidak formal]

INPUT:
korpus ujaran online: You 

In [14]:
# PILOT SAMPLE SELECTION

pilot_samples = (
    df.groupby("domain", group_keys=False)
      .sample(n=5, random_state=42)
      .reset_index(drop=True)
)

print("Jumlah sampel pilot:", len(pilot_samples))
print("\nBilangan sampel mengikut domain:")
print(pilot_samples["domain"].value_counts())

Jumlah sampel pilot: 25

Bilangan sampel mengikut domain:
domain
Campuran Kod     5
Penukaran Kod    5
Sentimen         5
Sentimen News    5
Terjemahan       5
Name: count, dtype: int64


In [15]:
for domain in PROMPTS:
    print("\n" + "=" * 70)
    print("DOMAIN:", domain)
    print("=" * 70)

    samples = pilot_samples[pilot_samples["domain"] == domain]

    for index, row in samples.iterrows():
        print(f"\nSample {index + 1}:")

        print("\nORIGINAL INPUT:")
        print(row["input"])

        print("\nCLEANED PROMPT:")
        print(build_prompt(row))

        print("\nREFERENCE:")
        print(row["reference"])


DOMAIN: Campuran Kod

Sample 1:

ORIGINAL INPUT:
{'korpus ujaran online': '31) Program exchange student ke Korea itu nampak interesting, tapi requirement untuk apply agak ketat dan kita perlukan strong recommendation daripada Head of Department.'}

CLEANED PROMPT:

Anda ialah pembantu yang menjana respons
berdasarkan konteks perbualan harian.

TUGAS:
Anda akan diberikan satu ujaran, soalan atau
mesej daripada seseorang.

Hasilkan DUA respons yang sesuai untuk membalas
input tersebut:

1. FORMAL:
   Gunakan Bahasa Melayu formal, baku dan sopan.

2. TIDAK FORMAL:
   Gunakan Bahasa Melayu santai dan semula jadi
   seperti perbualan harian.

ARAHAN PENTING:
- Fahami maksud dan konteks input terlebih dahulu.
- Jawab soalan atau berikan respons yang sesuai
  terhadap mesej yang diberikan.
- Jangan mengolah semula ayat input.
- Jangan menterjemahkan input.
- Jangan mengulang input sebagai jawapan.
- Pastikan kedua-dua respons benar-benar menjawab
  input yang diberikan.
- Kedua-dua respons m

In [16]:
# Ambil 5 sampel Campuran Kod
samples = pilot_samples[
    pilot_samples["domain"] == "Campuran Kod"
]

for _, row in samples.iterrows():

    print("\n" + "=" * 70)

    print("ORIGINAL INPUT:")
    print(row["input"])

    print("\nCLEANED PROMPT:")
    print(build_prompt(row))


ORIGINAL INPUT:
{'korpus ujaran online': '31) Program exchange student ke Korea itu nampak interesting, tapi requirement untuk apply agak ketat dan kita perlukan strong recommendation daripada Head of Department.'}

CLEANED PROMPT:

Anda ialah pembantu yang menjana respons
berdasarkan konteks perbualan harian.

TUGAS:
Anda akan diberikan satu ujaran, soalan atau
mesej daripada seseorang.

Hasilkan DUA respons yang sesuai untuk membalas
input tersebut:

1. FORMAL:
   Gunakan Bahasa Melayu formal, baku dan sopan.

2. TIDAK FORMAL:
   Gunakan Bahasa Melayu santai dan semula jadi
   seperti perbualan harian.

ARAHAN PENTING:
- Fahami maksud dan konteks input terlebih dahulu.
- Jawab soalan atau berikan respons yang sesuai
  terhadap mesej yang diberikan.
- Jangan mengolah semula ayat input.
- Jangan menterjemahkan input.
- Jangan mengulang input sebagai jawapan.
- Pastikan kedua-dua respons benar-benar menjawab
  input yang diberikan.
- Kedua-dua respons mesti menyampaikan maksud
  jawapa

In [17]:
def run_inference(prompt):
    start_time = time.time()

    try:
        response = ollama.generate(
            model=MODEL_NAME,
            prompt=prompt,
            options=GENERATION_CONFIG,
            stream=False
        )

        prediction = response["response"].strip()
        duration = time.time() - start_time

        return {
            "prediction": prediction,
            "duration": round(duration, 2),
            "status": "success",
            "error": None
        }

    except Exception as e:
        duration = time.time() - start_time

        return {
            "prediction": None,
            "duration": round(duration, 2),
            "status": "failed",
            "error": str(e)
        }

In [18]:
test_record = pilot_samples.iloc[0].to_dict()

test_prompt = build_prompt(test_record)

test_result = run_inference(test_prompt)

print("DOMAIN:", test_record["domain"])
print("\nPREDICTION:")
print(test_result["prediction"])

print("\nINFERENCE TIME:", test_result["duration"], "seconds")
print("STATUS:", test_result["status"])

DOMAIN: Campuran Kod

PREDICTION:
FORMAL: Saya faham kebimbangan anda. Walaupun syarat permohonan agak ketat, peluang untuk memperoleh surat cadangan yang kukuh daripada Ketua Jabatan masih terbuka selagi anda berhubung dengan beliau seawal mungkin. Saya cadangkan anda membuat temu janji dengan HoD minggu ini untuk membincangkan keperluan tersebut secara terperinci.

TIDAK FORMAL: Haa, memang betul requirement dia ketat sikit. Tapi jangan stress dulu, okay? Cuba jumpa HoD tu awal-awal, explain yang kita nak apply, minta dia tolong tulis recommendation letter yang strong. Kalau kita approach dia dengan prepared, biasanya dia sudi tolong. Still ada hope lah!

INFERENCE TIME: 34.87 seconds
STATUS: success


In [19]:
# RUN PILOT INFERENCE

pilot_results = []

for i, (_, row) in enumerate(pilot_samples.iterrows(), start=1):

    record = row.to_dict()
    prompt = build_prompt(record)

    print(f"\n[{i}/{len(pilot_samples)}] {record['domain']}")

    result = run_inference(prompt)

    pilot_result = {
        "pilot_id": i,
        "domain": record["domain"],
        "input": record["input"],
        "reference": record["reference"],
        "prediction": result["prediction"],
        "inference_time": result["duration"],
        "status": result["status"],
        "error": result["error"]
    }

    pilot_results.append(pilot_result)

    print("Status:", result["status"])
    print("Time:", result["duration"], "seconds")

    if result["prediction"]:
        print("Prediction:", result["prediction"])


[1/25] Campuran Kod
Status: success
Time: 22.11 seconds
Prediction: FORMAL: Saya faham kebimbangan anda. Walaupun syarat permohonan agak ketat, peluang untuk memperoleh surat cadangan yang kukuh daripada Ketua Jabatan masih terbuka selagi anda berhubung dengan beliau seawal mungkin. Saya cadangkan anda membuat temu janji dengan HoD minggu ini untuk membincangkan keperluan tersebut secara terperinci.

TIDAK FORMAL: Haa, memang betul requirement dia ketat sikit. Tapi jangan stress dulu, okay? Cuba jumpa HoD tu awal-awal, explain yang kita nak apply, minta dia tolong tulis recommendation letter yang strong. Kalau kita approach dia dengan prepared, biasanya dia sudi tolong. Still ada hope lah!

[2/25] Campuran Kod
Status: success
Time: 19.63 seconds
Prediction: FORMAL: Sila buat laporan kehilangan di balai polis terdekat terlebih dahulu bagi mendapatkan laporan polis. Selepas itu, hadir ke Jabatan Pendaftaran Negara (JPN) untuk membuat permohonan penggantian kad pengenalan dengan membawa 

In [20]:
# PILOT RESULTS AUDIT

pilot_df = pd.DataFrame(pilot_results)

print("TOTAL RECORDS:", len(pilot_df))

print("\nSTATUS:")
print(pilot_df["status"].value_counts(dropna=False))

print("\nDOMAIN DISTRIBUTION:")
print(pilot_df["domain"].value_counts())

print("\nMISSING PREDICTIONS:")
print(pilot_df["prediction"].isna().sum())

print("\nMISSING REFERENCES:")
print(pilot_df["reference"].isna().sum())

print("\nDUPLICATE PILOT IDs:")
print(pilot_df["pilot_id"].duplicated().sum())

print("\nAVERAGE INFERENCE TIME:")
print(round(pilot_df["inference_time"].mean(), 2), "seconds")

print("\nTOTAL INFERENCE TIME:")
print(round(pilot_df["inference_time"].sum() / 60, 2), "minutes")

TOTAL RECORDS: 25

STATUS:
status
success    25
Name: count, dtype: int64

DOMAIN DISTRIBUTION:
domain
Campuran Kod     5
Penukaran Kod    5
Sentimen         5
Sentimen News    5
Terjemahan       5
Name: count, dtype: int64

MISSING PREDICTIONS:
0

MISSING REFERENCES:
0

DUPLICATE PILOT IDs:
0

AVERAGE INFERENCE TIME:
20.01 seconds

TOTAL INFERENCE TIME:
8.34 minutes


In [21]:
for domain in PROMPTS:

    print("\n" + "=" * 90)
    print("DOMAIN:", domain)
    print("=" * 90)

    domain_results = pilot_df[
        pilot_df["domain"] == domain
    ]

    for _, row in domain_results.iterrows():

        print("\nINPUT:")
        print(row["input"])

        print("\nREFERENCE:")
        print(row["reference"])

        print("\nPREDICTION:")
        print(row["prediction"])

        print("\nINFERENCE TIME:")
        print(row["inference_time"], "seconds")

        print("-" * 90)


DOMAIN: Campuran Kod

INPUT:
{'korpus ujaran online': '31) Program exchange student ke Korea itu nampak interesting, tapi requirement untuk apply agak ketat dan kita perlukan strong recommendation daripada Head of Department.'}

REFERENCE:
{'ujaran tidak formal': 'Program tu memang interesting, tapi requirement dia tough. Recommendation HOD tu paling critical.', 'ujaran formal': 'Program itu memang menarik, tetapi keperluan dia tinggi. Cadangan ketua jabatan itu sangat penting'}

PREDICTION:
FORMAL: Saya faham kebimbangan anda. Walaupun syarat permohonan agak ketat, peluang untuk memperoleh surat cadangan yang kukuh daripada Ketua Jabatan masih terbuka selagi anda berhubung dengan beliau seawal mungkin. Saya cadangkan anda membuat temu janji dengan HoD minggu ini untuk membincangkan keperluan tersebut secara terperinci.

TIDAK FORMAL: Haa, memang betul requirement dia ketat sikit. Tapi jangan stress dulu, okay? Cuba jumpa HoD tu awal-awal, explain yang kita nak apply, minta dia tolong

In [22]:
# ==========================================
# LOAD FULL DATASET
# ==========================================

records = []

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)

print("DATASET LOADED")
print("=" * 50)

print("Total records:", len(df))
print("\nDomain distribution:")
print(df["domain"].value_counts())

assert len(df) == EXPECTED_RECORDS

print("\nDataset validation: PASS")

DATASET LOADED
Total records: 3448

Domain distribution:
domain
Sentimen News    980
Penukaran Kod    799
Terjemahan       712
Sentimen         682
Campuran Kod     275
Name: count, dtype: int64

Dataset validation: PASS


In [23]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS")
RESULT_DIR = BASE_DIR / "results" / "qwen"
PILOT_FILE = RESULT_DIR / "pilot_results.jsonl"

# Load recovered pilot results
with open(PILOT_FILE, "r", encoding="utf-8") as f:
    pilot_results = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

pilot_df = pd.DataFrame(pilot_results)

# Verify without displaying confidential data
assert len(pilot_df) == 25, "Pilot records tidak cukup!"
assert pilot_df["pilot_id"].nunique() == 25, "Duplicate pilot ID!"
assert pilot_df["prediction"].notna().all(), "Ada prediction kosong!"

print("PILOT DATA LOADED SUCCESSFULLY")
print("Total records:", len(pilot_df))
print("\nDomain distribution:")
print(pilot_df["domain"].value_counts())

PILOT DATA LOADED SUCCESSFULLY
Total records: 25

Domain distribution:
domain
Campuran Kod     5
Penukaran Kod    5
Sentimen         5
Sentimen News    5
Terjemahan       5
Name: count, dtype: int64


In [24]:
import re
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sacrebleu import corpus_bleu, corpus_chrf
from bert_score import score as bert_score

# ==========================================
# 1. CONFIGURATION
# ==========================================

OUTPUT_DIR = RESULT_DIR / "pilot_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["Positif", "Neutral", "Negatif"]

generation_domains = [
    "Campuran Kod",
    "Penukaran Kod",
    "Terjemahan"
]

classification_domains = [
    "Sentimen",
    "Sentimen News"
]

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def clean_label(value):
    if not isinstance(value, str):
        return None

    match = re.fullmatch(
        r"\s*(positif|neutral|negatif)\s*[\.\!]*\s*",
        value,
        re.IGNORECASE
    )

    return match.group(1).capitalize() if match else None


def extract_prediction_label(text):
    if not isinstance(text, str):
        return None

    return clean_label(text)


def get_reference_label(reference):
    if not isinstance(reference, dict):
        return clean_label(reference)

    # Reference classification should contain one label.
    for value in reference.values():
        label = clean_label(value)
        if label:
            return label

    return None


def get_text(value):
    if value is None:
        return None

    if isinstance(value, float) and np.isnan(value):
        return None

    text = str(value).strip()
    return text if text else None


def normalize_key(key):
    return re.sub(r"[^a-z]", "", str(key).lower())


def extract_section(prediction, section):

    if not isinstance(prediction, str):
        return None

    # Cari kedua-dua heading
    pattern = (
        r"(?im)^\s*(?:\*\*)?\s*"
        r"(TIDAK\s+FORMAL|FORMAL)"
        r"\s*(?:\*\*)?\s*:"
    )

    matches = list(re.finditer(pattern, prediction))

    if not matches:
        return None

    # Simpan heading mengikut kedudukan
    headings = []

    for match in matches:
        heading = re.sub(r"\s+", " ", match.group(1)).upper()

        if heading == "FORMAL":
            headings.append(("formal", match))

        elif heading == "TIDAK FORMAL":
            headings.append(("informal", match))

    # Cari bahagian yang diminta
    for i, (heading_type, match) in enumerate(headings):

        if heading_type != section:
            continue

        start = match.end()

        if i + 1 < len(headings):
            end = headings[i + 1][1].start()
        else:
            end = len(prediction)

        result = prediction[start:end].strip()

        # Bersihkan Markdown
        result = re.sub(r"^\s*[-*]\s*", "", result)
        result = result.strip()

        return result if result else None

    return None




def get_reference_for_section(reference, section):
    if not isinstance(reference, dict):
        return None

    for key, value in reference.items():

        normalized = normalize_key(key)

        if section == "formal":
            matches = (
                "formal" in normalized
                and "tidakformal" not in normalized
            )
        else:
            matches = "tidakformal" in normalized

        if matches:
            return get_text(value)

    return None


def extract_generation_pairs(domain, row):

    prediction = get_text(row["prediction"])
    reference = row["reference"]

    if not prediction or not isinstance(reference, dict):
        return []

    pairs = []

    if domain == "Terjemahan":

        ref = next(
            (
                get_text(value)
                for key, value in reference.items()
                if normalize_key(key) == "jawapanbahasasasaran"
            ),
            None
        )

        if ref:
            pairs.append({
                "pilot_id": row["pilot_id"],
                "domain": domain,
                "section": "translation",
                "prediction": prediction,
                "reference": ref
            })

    else:

        for section in ["formal", "informal"]:

            pred = extract_section(prediction, section)
            ref = get_reference_for_section(reference, section)

            if pred and ref:
                pairs.append({
                    "pilot_id": row["pilot_id"],
                    "domain": domain,
                    "section": section,
                    "prediction": pred,
                    "reference": ref
                })

    return pairs


# ==========================================
# 3. CLASSIFICATION EVALUATION
# ==========================================

classification_results = []
classification_details = []

for domain in classification_domains:

    subset = pilot_df[
        pilot_df["domain"] == domain
    ].copy()

    y_true = subset["reference"].apply(get_reference_label)
    y_pred = subset["prediction"].apply(
        extract_prediction_label
    )

    valid = y_true.notna() & y_pred.notna()

    true_valid = y_true[valid].tolist()
    pred_valid = y_pred[valid].tolist()

    metrics = {
        "domain": domain,
        "total_records": len(subset),
        "valid_records": len(true_valid),
        "invalid_records": len(subset) - len(true_valid),
        "accuracy": None,
        "precision_macro": None,
        "recall_macro": None,
        "macro_f1": None
    }

    if true_valid:

        metrics["accuracy"] = accuracy_score(
            true_valid, pred_valid
        )

        metrics["precision_macro"] = precision_score(
            true_valid, pred_valid,
            labels=LABELS,
            average="macro",
            zero_division=0
        )

        metrics["recall_macro"] = recall_score(
            true_valid, pred_valid,
            labels=LABELS,
            average="macro",
            zero_division=0
        )

        metrics["macro_f1"] = f1_score(
            true_valid, pred_valid,
            labels=LABELS,
            average="macro",
            zero_division=0
        )

    classification_results.append(metrics)

    for index, row in subset.iterrows():

        actual = y_true.loc[index]
        predicted = y_pred.loc[index]

        classification_details.append({
            "pilot_id": row["pilot_id"],
            "domain": domain,
            "actual_label": actual,
            "predicted_label": predicted,
            "valid": bool(
                pd.notna(actual) and pd.notna(predicted)
            ),
            "correct": (
                actual == predicted
                if pd.notna(actual) and pd.notna(predicted)
                else None
            )
        })


# ==========================================
# 4. GENERATION EVALUATION
# ==========================================

generation_pairs = []

for domain in generation_domains:

    subset = pilot_df[
        pilot_df["domain"] == domain
    ]

    for _, row in subset.iterrows():

        generation_pairs.extend(
            extract_generation_pairs(domain, row)
        )


generation_df = pd.DataFrame(generation_pairs)

print("\nGENERATION EXTRACTION AUDIT")
print("=" * 60)

if not generation_df.empty:
    print(
        generation_df.groupby(
            ["domain", "section"]
        ).size()
    )

print("\nTotal extracted pairs:", len(generation_df))

generation_results = []
generation_metric_details = []

if not generation_df.empty:

    for (domain, section), group in generation_df.groupby(
        ["domain", "section"]
    ):

        predictions = group["prediction"].tolist()
        references = group["reference"].tolist()

        bleu = corpus_bleu(
            predictions,
            [references]
        ).score

        chrfpp = corpus_chrf(
            predictions,
            [references],
            word_order=2
        ).score

        generation_results.append({
            "domain": domain,
            "section": section,
            "valid_pairs": len(predictions),
            "BLEU": bleu,
            "chrF++": chrfpp
        })


# ==========================================
# 5. BERTSCORE EVALUATION
# ==========================================

if not generation_df.empty:

    print("Calculating BERTScore...")
    print("Using multilingual BERT model.")

    P, R, F1 = bert_score(
        generation_df["prediction"].tolist(),
        generation_df["reference"].tolist(),
        model_type="bert-base-multilingual-cased",
        verbose=True,
        batch_size=4,
        device="cuda"
    )

    generation_df["BERTScore_P"] = P.cpu().numpy()
    generation_df["BERTScore_R"] = R.cpu().numpy()
    generation_df["BERTScore_F1"] = F1.cpu().numpy()

    bert_summary = (
        generation_df
        .groupby(["domain", "section"])
        ["BERTScore_F1"]
        .mean()
        .reset_index()
        .rename(
            columns={
                "BERTScore_F1": "BERTScore_F1_mean"
            }
        )
    )

    generation_metrics_df = pd.DataFrame(
        generation_results
    ).merge(
        bert_summary,
        on=["domain", "section"],
        how="left"
    )

else:
    generation_metrics_df = pd.DataFrame()


# ==========================================
# 6. SAVE RESULTS
# ==========================================

classification_df = pd.DataFrame(
    classification_results
)

classification_details_df = pd.DataFrame(
    classification_details
)

classification_df.to_csv(
    OUTPUT_DIR / "classification_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

classification_details_df.to_csv(
    OUTPUT_DIR / "classification_details.csv",
    index=False,
    encoding="utf-8-sig"
)

generation_metrics_df.to_csv(
    OUTPUT_DIR / "generation_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

generation_df.to_csv(
    OUTPUT_DIR / "generation_details.csv",
    index=False,
    encoding="utf-8-sig"
)


# ==========================================
# 7. FINAL SUMMARY
# ==========================================

print("\n" + "=" * 65)
print("CLASSIFICATION METRICS")
print("=" * 65)

print(classification_df.to_string(index=False))

print("\n" + "=" * 65)
print("GENERATION METRICS")
print("=" * 65)

print(generation_metrics_df.to_string(index=False))

print("\n" + "=" * 65)
print("EXTRACTION AUDIT")
print("=" * 65)

print("Original pilot records:", len(pilot_df))
print("Extracted generation pairs:", len(generation_df))

print("\nGeneration pairs by domain:")
if not generation_df.empty:
    print(
        generation_df.groupby(
            ["domain", "section"]
        ).size()
    )

print("\nFiles saved to:")
print(OUTPUT_DIR.resolve())


C:\AI\rocm-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



GENERATION EXTRACTION AUDIT
domain         section    
Campuran Kod   formal         5
               informal       1
Penukaran Kod  formal         5
               informal       5
Terjemahan     translation    5
dtype: int64

Total extracted pairs: 21
Calculating BERTScore...
Using multilingual BERT model.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33162.48it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/11 [00:00<?, ?it/s]C:\AI\rocm-env\Lib\site-packages\transformers\integrations\sdpa_attention.py:174: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\transformers\hip\sdp_utils.cpp:361.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
100%|██████████| 11/11 [00:01<00:00,  6.10it/s]


computing greedy matching.


100%|██████████| 6/6 [00:00<00:00, 17.32it/s]

done in 2.16 seconds, 9.74 sentences/sec

CLASSIFICATION METRICS
       domain  total_records  valid_records  invalid_records  accuracy  precision_macro  recall_macro  macro_f1
     Sentimen              5              5                0       1.0         0.666667      0.666667  0.666667
Sentimen News              5              5                0       0.4         0.444444      0.333333  0.355556

GENERATION METRICS
       domain     section  valid_pairs      BLEU    chrF++  BERTScore_F1_mean
 Campuran Kod      formal            5  1.073736 14.845514           0.678075
 Campuran Kod    informal            1  4.491576 33.613031           0.757732
Penukaran Kod      formal            5  2.557656 18.814673           0.676834
Penukaran Kod    informal            5  1.969645 11.690471           0.626317
   Terjemahan translation            5 25.621053 47.730769           0.837800

EXTRACTION AUDIT
Original pilot records: 25
Extracted generation pairs: 21

Generation pairs by domain:
domain

In [25]:
# ==========================================
# MANUAL VALIDATION - CLASSIFICATION
# ==========================================

print("\nCLASSIFICATION MANUAL VALIDATION")
print("=" * 70)

display(
    classification_details_df[
        [
            "pilot_id",
            "domain",
            "actual_label",
            "predicted_label",
            "valid",
            "correct"
        ]
    ].sort_values(["domain", "pilot_id"])
)


CLASSIFICATION MANUAL VALIDATION


,pilot_id,domain,actual_label,predicted_label,valid,correct
0,11,Sentimen,Positif,Positif,True,True
1,12,Sentimen,Negatif,Negatif,True,True
2,13,Sentimen,Positif,Positif,True,True
3,14,Sentimen,Positif,Positif,True,True
4,15,Sentimen,Positif,Positif,True,True
5,16,Sentimen News,Positif,Neutral,True,False
6,17,Sentimen News,Negatif,Neutral,True,False
7,18,Sentimen News,Negatif,Negatif,True,True
8,19,Sentimen News,Neutral,Neutral,True,True
9,20,Sentimen News,Neutral,Positif,True,False


In [26]:

# ==========================================
# PILOT SAMPLE SELECTION
# ==========================================

# Select 5 records from each domain
pilot_df = (
    df.groupby("domain", group_keys=False)
    .head(5)
    .copy()
    .reset_index(drop=True)
)

print("PILOT DATASET CREATED")
print("=" * 55)

print("Total pilot records:", len(pilot_df))

print("\nDomain distribution:")
print(pilot_df["domain"].value_counts())

print("\nColumns:")
print(pilot_df.columns.tolist())


PILOT DATASET CREATED
Total pilot records: 25

Domain distribution:
domain
Campuran Kod     5
Penukaran Kod    5
Terjemahan       5
Sentimen News    5
Sentimen         5
Name: count, dtype: int64

Columns:
['domain', 'input', 'reference', 'source_excel_row']


In [27]:
%pip install -q scikit-learn sacrebleu bert-score

Note: you may need to restart the kernel to use updated packages.


In [31]:
# import re
# import json
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score
# )
# from sacrebleu import corpus_bleu, corpus_chrf
# from bert_score import score as bert_score


# # ==========================================
# # 1. CONFIGURATION - QWEN V2 PILOT
# # ==========================================

# BASE_DIR = Path(
#     r"C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS"
# )

# RESULT_DIR = (
#     BASE_DIR
#     / "results"
#     / "qwen"
#     / "prompt_v2"
# )

# OUTPUT_DIR = RESULT_DIR / "pilot_evaluation"

# OUTPUT_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )

# LABELS = ["Positif", "Neutral", "Negatif"]

# LABELS = ["Positif", "Neutral", "Negatif"]

# generation_domains = [
#     "Campuran Kod",
#     "Penukaran Kod",
#     "Terjemahan"
# ]

# classification_domains = [
#     "Sentimen",
#     "Sentimen News"
# ]

# # ==========================================
# # 2. HELPER FUNCTIONS
# # ==========================================

# def extract_sentiment(text):
#     if not isinstance(text, str):
#         return None

#     match = re.search(
#         r"\b(positif|neutral|negatif)\b",
#         text,
#         re.IGNORECASE
#     )

#     if match:
#         return match.group(1).capitalize()

#     match = re.fullmatch(
#         r"\s*([ABC])[\.\)]?\s*",
#         text,
#         re.IGNORECASE
#     )

#     if match:
#         return {
#             "A": "Positif",
#             "B": "Neutral",
#             "C": "Negatif"
#         }[match.group(1).upper()]

#     return None


# def get_reference_label(reference):
#     if isinstance(reference, dict):
#         values = list(reference.values())
#     else:
#         values = [reference]

#     for value in values:
#         label = extract_sentiment(value)
#         if label:
#             return label

#     return None


# def get_text(value):
#     if value is None:
#         return None
#     if isinstance(value, float) and np.isnan(value):
#         return None
#     return str(value).strip()


# def extract_generation_pairs(domain, row):
#     """Return matching prediction-reference pairs."""

#     reference = row["reference"]
#     prediction = get_text(row["prediction"])

#     if not isinstance(reference, dict) or not prediction:
#         return []

#     pairs = []

#     if domain == "Terjemahan":
#         ref = next(
#             (v for v in reference.values() if get_text(v)),
#             None
#         )
#         if ref:
#             pairs.append((prediction, get_text(ref), "translation"))

#     else:
#         sections = {
#             "formal": r"FORMAL\s*:\s*(.*?)(?=TIDAK\s*FORMAL\s*:|$)",
#             "informal": r"TIDAK\s*FORMAL\s*:\s*(.*)$"
#         }

#         for section, pattern in sections.items():
#             ref = next(
#                 (
#                     get_text(v)
#                     for k, v in reference.items()
#                     if section in str(k).lower() and get_text(v)
#                 ),
#                 None
#             )

#             if ref:
#                 match = re.search(
#                     pattern,
#                     prediction,
#                     re.IGNORECASE | re.DOTALL
#                 )

#                 pred = match.group(1).strip() if match else None

#                 if pred:
#                     pairs.append((pred, ref, section))

#     return pairs


# # ==========================================
# # 3. CLASSIFICATION EVALUATION
# # ==========================================

# classification_results = []
# classification_details = []

# for domain in classification_domains:

#     subset = pilot_df[pilot_df["domain"] == domain].copy()

#     y_true = subset["reference"].apply(get_reference_label)
#     y_pred = subset["prediction"].apply(extract_sentiment)

#     valid = y_true.notna() & y_pred.notna()

#     true_valid = y_true[valid].tolist()
#     pred_valid = y_pred[valid].tolist()

#     if len(true_valid) == 0:
#         print(f"WARNING: No valid labels for {domain}")
#         continue

#     metrics = {
#         "domain": domain,
#         "total_records": len(subset),
#         "valid_records": len(true_valid),
#         "invalid_records": len(subset) - len(true_valid),
#         "accuracy": accuracy_score(true_valid, pred_valid),
#         "precision_macro": precision_score(
#             true_valid, pred_valid,
#             labels=LABELS, average="macro", zero_division=0
#         ),
#         "recall_macro": recall_score(
#             true_valid, pred_valid,
#             labels=LABELS, average="macro", zero_division=0
#         ),
#         "macro_f1": f1_score(
#             true_valid, pred_valid,
#             labels=LABELS, average="macro", zero_division=0
#         )
#     }

#     classification_results.append(metrics)

#     for idx, (_, row) in enumerate(subset.iterrows()):
#         classification_details.append({
#             "domain": domain,
#             "prediction": row["prediction"],
#             "reference": row["reference"],
#             "actual_label": y_true.loc[row.name],
#             "predicted_label": y_pred.loc[row.name],
#             "correct": (
#                 y_true.loc[row.name] == y_pred.loc[row.name]
#                 if valid.loc[row.name] else None
#             )
#         })


# # ==========================================
# # 4. TEXT GENERATION EVALUATION
# # ==========================================

# generation_pairs = []

# for domain in generation_domains:

#     subset = pilot_df[pilot_df["domain"] == domain]

#     for _, row in subset.iterrows():

#         pairs = extract_generation_pairs(domain, row)

#         for prediction, reference, section in pairs:
#             generation_pairs.append({
#                 "domain": domain,
#                 "section": section,
#                 "prediction": prediction,
#                 "reference": reference
#             })


# generation_df = pd.DataFrame(generation_pairs)
# generation_results = []

# if not generation_df.empty:

#     for (domain, section), group in generation_df.groupby(
#         ["domain", "section"]
#     ):

#         predictions = group["prediction"].tolist()
#         references = group["reference"].tolist()

#         bleu = corpus_bleu(
#             predictions,
#             [references]
#         ).score

#         chrf = corpus_chrf(
#             predictions,
#             [references]
#         ).score

#         generation_results.append({
#             "domain": domain,
#             "section": section,
#             "valid_pairs": len(predictions),
#             "BLEU": bleu,
#             "chrF++": chrf
#         })

#     # BERTScore: English model may not be optimal for Malay.
#     # Use multilingual model for better language coverage.
#     P, R, F1 = bert_score(
#         generation_df["prediction"].tolist(),
#         generation_df["reference"].tolist(),
#         model_type="bert-base-multilingual-cased",
#         lang="en",
#         verbose=False,
#         batch_size=4
#     )

#     generation_df["BERTScore_P"] = P.tolist()
#     generation_df["BERTScore_R"] = R.tolist()
#     generation_df["BERTScore_F1"] = F1.tolist()

#     bert_summary = (
#         generation_df
#         .groupby(["domain", "section"])["BERTScore_F1"]
#         .mean()
#         .reset_index()
#         .rename(columns={"BERTScore_F1": "BERTScore_F1_mean"})
#     )

#     generation_metrics_df = pd.DataFrame(generation_results)
#     generation_metrics_df = generation_metrics_df.merge(
#         bert_summary,
#         on=["domain", "section"],
#         how="left"
#     )

# else:
#     generation_metrics_df = pd.DataFrame()


# # ==========================================
# # 5. SAVE RESULTS
# # ==========================================

# classification_df = pd.DataFrame(classification_results)
# classification_details_df = pd.DataFrame(classification_details)

# classification_df.to_csv(
#     OUTPUT_DIR / "classification_metrics.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# classification_details_df.to_csv(
#     OUTPUT_DIR / "classification_details.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# generation_metrics_df.to_csv(
#     OUTPUT_DIR / "generation_metrics.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# generation_df.to_csv(
#     OUTPUT_DIR / "generation_details.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# # ==========================================
# # 6. DISPLAY SUMMARY
# # ==========================================

# print("\n" + "=" * 70)
# print("CLASSIFICATION EVALUATION")
# print("=" * 70)
# print(classification_df.to_string(index=False))

# print("\n" + "=" * 70)
# print("TEXT GENERATION EVALUATION")
# print("=" * 70)
# print(generation_metrics_df.to_string(index=False))

# print("\n" + "=" * 70)
# print("EVALUATION FILES SAVED")
# print("=" * 70)
# print("Output directory:", OUTPUT_DIR.resolve())

# print("\nPilot records:", len(pilot_df))
# print("Classification comparisons:", len(classification_details_df))
# print("Generation comparisons:", len(generation_df))


In [32]:
### INFERENCE / PREDICTION PROCESS ###

In [33]:
%pip install -q ollama pandas numpy tqdm scikit-learn sacrebleu bert-score

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import re
import json
import time
import random
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime

import ollama

# ==========================================
# 1. PROJECT CONFIGURATION
# ==========================================

BASE_DIR = Path(
    r"C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS"
)

DATASET_FILE = (
    BASE_DIR /
    "malay_cleaned_dataset" /
    "malay_evaluation_dataset.jsonl"
)

RESULT_DIR = BASE_DIR / "results" / "qwen" / "prompt_v2"

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ==========================================
# 2. MODEL CONFIGURATION
# ==========================================

MODEL_NAME = "qwen3.8:27b"

TEMPERATURE = 0
SEED = 42
NUM_PREDICT = 1024

GENERATION_OPTIONS = {
    "temperature": TEMPERATURE,
    "seed": SEED,
    "num_predict": NUM_PREDICT
}

# ==========================================
# 3. OUTPUT FILES
# ==========================================

FULL_RESULTS_FILE = (
    RESULT_DIR / "full_results_qwen_v2.jsonl"
)

FULL_PROGRESS_FILE = (
    RESULT_DIR / "full_progress_qwen_v2.jsonl"
)

FULL_FAILED_FILE = (
    RESULT_DIR / "full_failed_qwen_v2.jsonl"
)

# ==========================================
# 4. GENERAL CONFIGURATION
# ==========================================

EXPECTED_RECORDS = 3448

SAVE_EVERY = 1

print("CONFIGURATION LOADED")
print("=" * 55)
print("Model:", MODEL_NAME)
print("Dataset:", DATASET_FILE)
print("Expected records:", EXPECTED_RECORDS)
print("Temperature:", TEMPERATURE)
print("Seed:", SEED)
print("Num predict:", NUM_PREDICT)

assert DATASET_FILE.exists(), "Dataset file not found!"

print("\nDataset file verified.")

CONFIGURATION LOADED
Model: qwen3.8:27b
Dataset: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\malay_cleaned_dataset\malay_evaluation_dataset.jsonl
Expected records: 3448
Temperature: 0
Seed: 42
Num predict: 1024

Dataset file verified.


In [8]:
# ==========================================
# RELOAD FULL DATASET - QWEN V2
# ==========================================

records = []

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)

EXPECTED_RECORDS = 3448

assert len(df) == EXPECTED_RECORDS

# Recreate record IDs
df["record_id"] = (
    df["domain"].astype(str)
    + "_"
    + df["source_excel_row"].astype(str)
)

assert df["record_id"].is_unique

print("DATASET RELOADED SUCCESSFULLY")
print("Total records:", len(df))
print("\nDomain distribution:")
print(df["domain"].value_counts())


DATASET RELOADED SUCCESSFULLY
Total records: 3448

Domain distribution:
domain
Sentimen News    980
Penukaran Kod    799
Terjemahan       712
Sentimen         682
Campuran Kod     275
Name: count, dtype: int64


In [9]:
# ==========================================
# PROMPT CONFIGURATION
# ==========================================

PROMPTS = {

"Campuran Kod": """
Anda ialah pembantu yang menjana respons
berdasarkan konteks perbualan harian.

TUGAS:
Anda akan diberikan satu ujaran, soalan atau
mesej daripada seseorang.

Hasilkan DUA respons yang sesuai untuk membalas
input tersebut:

1. FORMAL:
   Gunakan Bahasa Melayu formal, baku dan sopan.

2. TIDAK FORMAL:
   Gunakan Bahasa Melayu santai dan semula jadi
   seperti perbualan harian.

ARAHAN PENTING:
- Fahami maksud dan konteks input terlebih dahulu.
- Jawab soalan atau berikan respons yang sesuai
  terhadap mesej yang diberikan.
- Jangan mengolah semula ayat input.
- Jangan menterjemahkan input.
- Jangan mengulang input sebagai jawapan.
- Pastikan kedua-dua respons benar-benar menjawab
  input yang diberikan.
- Kedua-dua respons mesti menyampaikan maksud
  jawapan yang sama tetapi menggunakan gaya
  bahasa yang berbeza.
- Jangan tambah penerangan atau ulasan.

FORMAT OUTPUT:
FORMAL: [Jawapan formal]
TIDAK FORMAL: [Jawapan tidak formal]

INPUT:
{input}
""",

"Penukaran Kod": """
Anda ialah pembantu yang menjana respons
berdasarkan konteks komunikasi profesional.

TUGAS:
Anda akan diberikan satu ujaran, soalan atau
mesej daripada seseorang.

Hasilkan DUA respons yang sesuai untuk membalas
input tersebut:

1. FORMAL:
   Gunakan Bahasa Melayu yang formal, baku,
   profesional dan sopan.

2. TIDAK FORMAL:
   Gunakan gaya bahasa yang lebih santai,
   semula jadi dan fleksibel, tetapi masih
   sesuai dalam konteks komunikasi profesional.
   Penggunaan Bahasa Melayu dan Bahasa Inggeris
   secara bercampur dibenarkan jika sesuai.

ARAHAN PENTING:
- Fahami maksud dan konteks input terlebih dahulu.
- Jawab soalan atau berikan respons yang sesuai
  terhadap mesej yang diberikan.
- Jangan mengolah semula ayat input.
- Jangan menterjemahkan input.
- Jangan mengulang input sebagai jawapan.
- Pastikan kedua-dua respons menjawab input.
- Kedua-dua respons mesti menyampaikan maksud
  jawapan yang sama tetapi menggunakan gaya
  bahasa yang berbeza.
- Jangan tambah penerangan atau ulasan.

FORMAT OUTPUT:
FORMAL: [Jawapan formal]
TIDAK FORMAL: [Jawapan tidak formal]

INPUT:
{input}
""",

"Terjemahan": """
Anda ialah penterjemah profesional
Bahasa Melayu dan Bahasa Inggeris.

TUGAS:
Terjemahkan teks yang diberikan ke bahasa
yang berbeza daripada bahasa sumber.

ARAHAN PENTERJEMAHAN:
- Jika teks sumber dalam Bahasa Inggeris,
  terjemahkan kepada Bahasa Melayu.
- Jika teks sumber dalam Bahasa Melayu,
  terjemahkan kepada Bahasa Inggeris.

ARAHAN PENTING:
- Kenal pasti bahasa sumber sebelum menterjemah.
- Gunakan bahasa sasaran yang betul.
- Kekalkan maksud asal dan konteks ayat.
- Pastikan semua maklumat penting dikekalkan.
- Gunakan tatabahasa dan struktur ayat
  yang betul serta semula jadi.
- Terjemahkan berdasarkan maksud dan konteks,
  bukan secara literal semata-mata.
- Jangan tambah atau buang maklumat penting.
- Jangan berikan penerangan atau ulasan.
- Jangan ulang teks sumber.

FORMAT OUTPUT:
Berikan hasil terjemahan sahaja.

TEKS SUMBER:
{input}
""",

    "Sentimen": """
Anda ialah penganalisis sentimen bahasa Melayu.

Jawab soalan berdasarkan pilihan jawapan yang diberikan.

Berikan jawapan dalam bentuk label sahaja:
Positif, Neutral atau Negatif.

Soalan:
{input}
""",


"Sentimen News": """
Anda ialah penganalisis sentimen berita Bahasa Melayu.

TUGAS:
Analisis teks berita yang diberikan dan tentukan
sifat sentimennya.

PILIH SATU LABEL SAHAJA:
- positif
- neutral
- negatif

PANDUAN:
- Positif: Berita menyampaikan perkembangan,
  keadaan atau peristiwa yang bersifat positif.
- Negatif: Berita menyampaikan perkembangan,
  keadaan atau peristiwa yang bersifat negatif.
- Neutral: Berita bersifat informatif atau
  melaporkan fakta tanpa sentimen positif
  atau negatif yang jelas.

ARAHAN PENTING:
- Nilai sentimen berdasarkan keseluruhan teks berita.
- Jangan membuat andaian berdasarkan satu perkataan sahaja.
- Jangan berikan penerangan atau alasan.
- Jangan keluarkan lebih daripada satu label.

FORMAT OUTPUT:
Berikan satu label sahaja: positif, neutral atau negatif.

TEKS BERITA:
{input}
"""

}

In [10]:
# ==========================================
# BUILD PROMPT - QWEN V2
# ==========================================


def build_prompt(row):

    domain = row["domain"]

    if domain not in PROMPTS:
        raise ValueError(
            f"Unknown domain: {domain}"
        )

    raw_input = row["input"]

    if isinstance(raw_input, dict):
        input_text = "\n".join(
            str(value).strip()
            for value in raw_input.values()
            if value is not None
        )
    else:
        input_text = str(raw_input).strip()

    if domain == "Campuran Kod":
        input_text = re.sub(
            r"^\s*\d+\)\s*",
            "",
            input_text
        )

    return PROMPTS[domain].format(
        input=input_text
    )



print("Qwen V2 prompt builder ready.")


Qwen V2 prompt builder ready.


In [11]:
# ==========================================
# QWEN INFERENCE
# ==========================================

def run_inference(prompt):

    start_time = time.time()

    response = ollama.generate(
        model=MODEL_NAME,
        prompt=prompt,
        options=GENERATION_OPTIONS
    )

    prediction = response["response"].strip()

    elapsed_time = time.time() - start_time

    return {
        "prediction": prediction,
        "inference_time": elapsed_time,
        "status": "success"
    }

print("Inference function ready.")

Inference function ready.


In [12]:

# ==========================================
# QWEN V2 FINAL PRE-FLIGHT CHECK
# ==========================================

print("QWEN V2 FINAL PRE-FLIGHT")
print("=" * 60)

# Dataset
assert len(df) == 3448
assert df["record_id"].is_unique

# All domains have prompts
assert set(df["domain"].unique()) == set(PROMPTS.keys())

# Prompt generation
for domain in PROMPTS:

    sample = df[df["domain"] == domain].iloc[0]

    prompt = build_prompt(sample)

    assert isinstance(prompt, str)
    assert len(prompt.strip()) > 0

    print(f"{domain}: PROMPT PASS")

# Output directory
assert RESULT_DIR.name == "prompt_v2"

print("\nModel:", MODEL_NAME)
print("Dataset records:", len(df))
print("Output directory:", RESULT_DIR.resolve())

print("\nResults file:", FULL_RESULTS_FILE.name)
print("Failed file:", FULL_FAILED_FILE.name)

# Check inference function availability
assert callable(run_inference)

print("\nInference function: READY")
print("\nALL PRE-FLIGHT CHECKS PASSED")


QWEN V2 FINAL PRE-FLIGHT
Campuran Kod: PROMPT PASS
Penukaran Kod: PROMPT PASS
Terjemahan: PROMPT PASS
Sentimen: PROMPT PASS
Sentimen News: PROMPT PASS

Model: qwen3.8:27b
Dataset records: 3448
Output directory: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\results\qwen\prompt_v2

Results file: full_results_qwen_v2.jsonl
Failed file: full_failed_qwen_v2.jsonl

Inference function: READY

ALL PRE-FLIGHT CHECKS PASSED


In [13]:

# ==========================================
# FULL QWEN BENCHMARK WITH CHECKPOINT & RESUME
# ==========================================

# 1. OUTPUT CONFIGURATION

FULL_RESULTS_FILE = RESULT_DIR / "full_results_qwen_v2.jsonl"
FULL_FAILED_FILE = RESULT_DIR / "full_failed_qwen_v2.jsonl"


# ==========================================
# 2. CHECK RECORD IDENTIFIERS
# ==========================================

assert "source_excel_row" in df.columns
assert "domain" in df.columns

assert df["source_excel_row"].notna().all()
assert df["domain"].notna().all()

# Create unique ID using domain + Excel row
df["record_id"] = (
    df["domain"].astype(str)
    + "_"
    + df["source_excel_row"].astype(str)
)

assert df["record_id"].is_unique, (
    "Duplicate record IDs still detected!"
)

print("Record ID validation passed.")
print("Total records:", len(df))
print("Unique record IDs:", df["record_id"].nunique())

display(
    df[["record_id", "domain", "source_excel_row"]].head(10)
)


# ==========================================
# 3. LOAD PREVIOUS SUCCESSFUL RESULTS
# ==========================================

def load_completed_results():

    completed = {}

    if FULL_RESULTS_FILE.exists():

        with open(FULL_RESULTS_FILE, "r", encoding="utf-8") as f:

            for line in f:

                if not line.strip():
                    continue

                result = json.loads(line)

                # Prevent mixing old result formats
                required_fields = [
                    "record_id",
                    "source_excel_row",
                    "domain",
                    "status"
                ]

                for field in required_fields:
                    if field not in result:
                        raise ValueError(
                            f"Missing field '{field}' in existing results. "
                            "Check the output file before continuing."
                        )

                if result.get("status") == "success":

                    record_id = str(result["record_id"])

                    # Validate record ID consistency
                    expected_id = (
                        str(result["domain"])
                        + "_"
                        + str(result["source_excel_row"])
                    )

                    if record_id != expected_id:
                        raise ValueError(
                            f"Record ID mismatch: {record_id}"
                        )

                    completed[record_id] = result

    return completed


completed_results = load_completed_results()


# ==========================================
# 4. VALIDATE EXISTING RESULTS
# ==========================================

valid_ids = set(df["record_id"].astype(str))

unexpected_ids = (
    set(completed_results.keys()) - valid_ids
)

assert not unexpected_ids, (
    "Output file contains records from another dataset!"
)

print("Previously completed:", len(completed_results))


# ==========================================
# 5. PREPARE PENDING RECORDS
# ==========================================

pending_rows = []

for index, row in df.iterrows():

    record_id = str(row["record_id"])

    if record_id not in completed_results:
        pending_rows.append((index, row, record_id))


print("Pending records:", len(pending_rows))
print("Total dataset:", len(df))


# ==========================================
# 6. PRE-INFERENCE CHECK
# ==========================================

print("\nPRE-INFERENCE CHECK")
print("=" * 50)

print("Total records:", len(df))
print("Previously completed:", len(completed_results))
print("Pending records:", len(pending_rows))

assert len(df) == 3448

assert (
    len(completed_results) + len(pending_rows)
    == len(df)
)

print("\nAll checks passed.")


# ==========================================
# 7. RUN FULL INFERENCE WITH ETA
# ==========================================

start_full = time.time()

session_success = 0
session_failed = 0
processed_count = 0

total_pending = len(pending_rows)

# Store inference time for each successful record
inference_times = []

with open(
    FULL_RESULTS_FILE,
    "a",
    encoding="utf-8"
) as result_file, open(
    FULL_FAILED_FILE,
    "a",
    encoding="utf-8"
) as failed_file:

    progress_bar = tqdm(
        pending_rows,
        total=total_pending,
        desc="Qwen Full Benchmark"
    )

    for index, row, record_id in progress_bar:

        try:

            prompt = build_prompt(row)

            output = run_inference(prompt)

            result = {
                "record_id": record_id,
                "source_excel_row": row["source_excel_row"],
                "domain": row["domain"],
                "input": row["input"],
                "reference": row["reference"],
                "prediction": output["prediction"],
                "inference_time": output["inference_time"],
                "status": "success"
            }

            # Save successful result immediately
            result_file.write(
                json.dumps(result, ensure_ascii=False) + "\n"
            )

            result_file.flush()

            completed_results[record_id] = result

            session_success += 1

            inference_times.append(
                output["inference_time"]
            )

        except Exception as e:

            failed_result = {
                "record_id": record_id,
                "source_excel_row": row["source_excel_row"],
                "domain": row["domain"],
                "error": str(e),
                "status": "failed"
            }

            # Save failed result
            failed_file.write(
                json.dumps(failed_result, ensure_ascii=False) + "\n"
            )

            failed_file.flush()

            session_failed += 1

        processed_count += 1

        # Update ETA every 10 records and at the end
        if (
            processed_count % 10 == 0
            or processed_count == total_pending
        ):

            elapsed = time.time() - start_full

            avg_time = (
                np.mean(inference_times)
                if inference_times
                else 0
            )

            remaining = total_pending - processed_count

            eta_seconds = avg_time * remaining

            progress_bar.set_postfix({

                "Success": session_success,

                "Failed": session_failed,

                "Avg/record": f"{avg_time:.1f}s",

                "ETA": str(
                    pd.to_timedelta(
                        int(eta_seconds),
                        unit="s"
                    )
                )
            })

            print(
                f"\nProgress: {processed_count}/{total_pending}"

                f"\nSuccessful: {session_success}"

                f"\nFailed: {session_failed}"

                f"\nElapsed: "
                f"{pd.to_timedelta(int(elapsed), unit='s')}"

                f"\nEstimated remaining: "
                f"{pd.to_timedelta(int(eta_seconds), unit='s')}"

                f"\nEstimated completion: "
                f"{datetime.now() + pd.to_timedelta(eta_seconds, unit='s')}"
            )


# ==========================================
# 8. FINAL SUMMARY
# ==========================================

elapsed_total = time.time() - start_full

print("\n" + "=" * 60)
print("QWEN FULL BENCHMARK SUMMARY")
print("=" * 60)

print("Total dataset:", len(df))

print("Successful this session:", session_success)

print("Failed this session:", session_failed)

print("Total successful:", len(completed_results))

print(
    "Remaining records:",
    len(df) - len(completed_results)
)

print(
    "Elapsed:",
    pd.to_timedelta(int(elapsed_total), unit="s")
)

if inference_times:

    print(
        "Average inference time:",
        round(np.mean(inference_times), 2),
        "seconds"
    )

print("\nResults saved:", FULL_RESULTS_FILE)

print("Failed records saved:", FULL_FAILED_FILE)


# ==========================================
# 9. FINAL VALIDATION
# ==========================================

print("\nFINAL VALIDATION")
print("=" * 60)

assert len(completed_results) <= len(df)

print(
    "Successful records:",
    len(completed_results),
    "/",
    len(df)
)

print(
    "Completion rate:",
    round(
        len(completed_results) / len(df) * 100,
        2
    ),
    "%"
)

print("\nBenchmark inference process finished.")


Record ID validation passed.
Total records: 3448
Unique record IDs: 3448


,record_id,domain,source_excel_row
0,Campuran Kod_3,Campuran Kod,3
1,Campuran Kod_4,Campuran Kod,4
2,Campuran Kod_5,Campuran Kod,5
3,Campuran Kod_6,Campuran Kod,6
4,Campuran Kod_7,Campuran Kod,7
5,Campuran Kod_8,Campuran Kod,8
6,Campuran Kod_9,Campuran Kod,9
7,Campuran Kod_10,Campuran Kod,10
8,Campuran Kod_11,Campuran Kod,11
9,Campuran Kod_12,Campuran Kod,12


Previously completed: 0
Pending records: 3448
Total dataset: 3448

PRE-INFERENCE CHECK
Total records: 3448
Previously completed: 0
Pending records: 3448

All checks passed.


Qwen Full Benchmark:   0%|          | 10/3448 [04:06<20:35:55, 21.57s/it, Success=10, Failed=0, Avg/record=24.6s, ETA=0 days 23:31:32]


Progress: 10/3448
Successful: 10
Failed: 0
Elapsed: 0 days 00:04:06
Estimated remaining: 0 days 23:31:32
Estimated completion: 2026-11-14 09:11:46.541300


Qwen Full Benchmark:   1%|          | 20/3448 [08:22<22:15:55, 23.38s/it, Success=20, Failed=0, Avg/record=25.1s, ETA=0 days 23:54:05]

... (683 outputs left out)

Qwen Full Benchmark: 100%|█████████▉| 3440/3448 [18:07:29<01:09,  8.68s/it, Success=3440, Failed=0, Avg/record=19.0s, ETA=0 days 00:02:31]


Progress: 3440/3448
Successful: 3440
Failed: 0
Elapsed: 0 days 18:07:29
Estimated remaining: 0 days 00:02:31
Estimated completion: 2026-11-14 03:46:09.418526


Qwen Full Benchmark: 100%|██████████| 3448/3448 [18:08:20<00:00, 18.94s/it, Success=3448, Failed=0, Avg/record=18.9s, ETA=0 days 00:00:00]


Progress: 3448/3448
Successful: 3448
Failed: 0
Elapsed: 0 days 18:08:20
Estimated remaining: 0 days 00:00:00
Estimated completion: 2026-11-14 03:44:28.368293

QWEN FULL BENCHMARK SUMMARY
Total dataset: 3448
Successful this session: 3448
Failed this session: 0
Total successful: 3448
Remaining records: 0
Elapsed: 0 days 18:08:20
Average inference time: 18.94 seconds

Results saved: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\results\qwen\prompt_v2\full_results_qwen_v2.jsonl
Failed records saved: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\results\qwen\prompt_v2\full_failed_qwen_v2.jsonl

FINAL VALIDATION
Successful records: 3448 / 3448
Completion rate: 100.0 %

Benchmark inference process finished.


In [14]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

# ==========================================
# 1. FILE PATHS
# ==========================================

BASE_DIR = Path.cwd()

RESULT_FILE = (
    BASE_DIR / "results" / "qwen"
    / "full_results_qwen.jsonl"
)

DATASET_FILE = (
    BASE_DIR / "malay_cleaned_dataset"
    / "malay_evaluation_dataset.jsonl"
)

# ==========================================
# 2. LOAD FULL RESULTS
# ==========================================

results = []

with open(RESULT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

results_df = pd.DataFrame(results)

print("=" * 60)
print("QWEN FULL RESULTS AUDIT")
print("=" * 60)

print("Total predictions:", len(results_df))
print("Columns:", results_df.columns.tolist())

if "domain" in results_df.columns:
    print("\nDomain distribution:")
    print(results_df["domain"].value_counts())

if "status" in results_df.columns:
    print("\nInference status:")
    print(results_df["status"].value_counts(dropna=False))

# Check missing values without displaying private data
print("\nMissing values:")
print(results_df.isna().sum())

# ==========================================
# 3. INSPECT RECORD STRUCTURE
# ==========================================

print("\nFirst record structure:")
for key, value in results[0].items():
    print(
        key,
        "->",
        type(value).__name__
    )

# ==========================================
# 4. LOAD ORIGINAL DATASET
# ==========================================

dataset_records = []

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            dataset_records.append(json.loads(line))

print("\nOriginal dataset records:", len(dataset_records))

print("\nDataset domain distribution:")
print(
    pd.Series([r["domain"] for r in dataset_records])
    .value_counts()
)

QWEN FULL RESULTS AUDIT
Total predictions: 3448
Columns: ['record_id', 'source_excel_row', 'domain', 'input', 'reference', 'prediction', 'inference_time', 'status']

Domain distribution:
domain
Sentimen News    980
Penukaran Kod    799
Terjemahan       712
Sentimen         682
Campuran Kod     275
Name: count, dtype: int64

Inference status:
status
success    3448
Name: count, dtype: int64

Missing values:
record_id           0
source_excel_row    0
domain              0
input               0
reference           0
prediction          0
inference_time      0
status              0
dtype: int64

First record structure:
record_id -> str
source_excel_row -> int
domain -> str
input -> dict
reference -> dict
prediction -> str
inference_time -> float
status -> str

Original dataset records: 3448

Dataset domain distribution:
Sentimen News    980
Penukaran Kod    799
Terjemahan       712
Sentimen         682
Campuran Kod     275
Name: count, dtype: int64


In [15]:
### DATA AUDIT / PRE-EVALUATION VALIDATION ###

In [16]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

# ==========================================
# 1. FILE PATHS
# ==========================================

BASE_DIR = Path.cwd()

RESULT_FILE = (
    BASE_DIR / "results" / "qwen"
    / "full_results_qwen.jsonl"
)

DATASET_FILE = (
    BASE_DIR / "malay_cleaned_dataset"
    / "malay_evaluation_dataset.jsonl"
)

# ==========================================
# 2. LOAD FULL RESULTS
# ==========================================

results = []

with open(RESULT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

results_df = pd.DataFrame(results)

print("=" * 60)
print("QWEN FULL RESULTS AUDIT")
print("=" * 60)

print("Total predictions:", len(results_df))
print("Columns:", results_df.columns.tolist())

if "domain" in results_df.columns:
    print("\nDomain distribution:")
    print(results_df["domain"].value_counts())

if "status" in results_df.columns:
    print("\nInference status:")
    print(results_df["status"].value_counts(dropna=False))

# Check missing values without displaying private data
print("\nMissing values:")
print(results_df.isna().sum())

# ==========================================
# 3. INSPECT RECORD STRUCTURE
# ==========================================

print("\nFirst record structure:")
for key, value in results[0].items():
    print(
        key,
        "->",
        type(value).__name__
    )

# ==========================================
# 4. LOAD ORIGINAL DATASET
# ==========================================

dataset_records = []

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            dataset_records.append(json.loads(line))

print("\nOriginal dataset records:", len(dataset_records))

print("\nDataset domain distribution:")
print(
    pd.Series([r["domain"] for r in dataset_records])
    .value_counts()
)

QWEN FULL RESULTS AUDIT
Total predictions: 3448
Columns: ['record_id', 'source_excel_row', 'domain', 'input', 'reference', 'prediction', 'inference_time', 'status']

Domain distribution:
domain
Sentimen News    980
Penukaran Kod    799
Terjemahan       712
Sentimen         682
Campuran Kod     275
Name: count, dtype: int64

Inference status:
status
success    3448
Name: count, dtype: int64

Missing values:
record_id           0
source_excel_row    0
domain              0
input               0
reference           0
prediction          0
inference_time      0
status              0
dtype: int64

First record structure:
record_id -> str
source_excel_row -> int
domain -> str
input -> dict
reference -> dict
prediction -> str
inference_time -> float
status -> str

Original dataset records: 3448

Dataset domain distribution:
Sentimen News    980
Penukaran Kod    799
Terjemahan       712
Sentimen         682
Campuran Kod     275
Name: count, dtype: int64


In [17]:
### VALIDATE RECORD MATCHING ###

In [18]:
import json
from collections import Counter

# ==========================================
# RECORD MATCHING VALIDATION
# ==========================================

def canonical_json(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True
    )

def make_key(record):
    return (
        record["domain"],
        record["source_excel_row"]
    )

# Create lookup dictionaries
original_lookup = {
    make_key(r): r for r in dataset_records
}

result_lookup = {
    make_key(r): r for r in results
}

# Check duplicate keys
original_keys = [make_key(r) for r in dataset_records]
result_keys = [make_key(r) for r in results]

duplicate_original = len(original_keys) - len(set(original_keys))
duplicate_results = len(result_keys) - len(set(result_keys))

# Compare records
matched = 0
mismatched = 0
missing_results = 0
extra_results = 0

for key, original in original_lookup.items():

    result = result_lookup.get(key)

    if result is None:
        missing_results += 1
        continue

    same_input = (
        canonical_json(original["input"])
        == canonical_json(result["input"])
    )

    same_reference = (
        canonical_json(original["reference"])
        == canonical_json(result["reference"])
    )

    if same_input and same_reference:
        matched += 1
    else:
        mismatched += 1

extra_results = len(
    set(result_lookup) - set(original_lookup)
)

# ==========================================
# DISPLAY AUDIT SUMMARY
# ==========================================

print("=" * 60)
print("RECORD MATCHING VALIDATION")
print("=" * 60)

print("Original records:", len(dataset_records))
print("Prediction records:", len(results))

print("\nDuplicate original keys:", duplicate_original)
print("Duplicate prediction keys:", duplicate_results)

print("\nMatched records:", matched)
print("Mismatched records:", mismatched)
print("Missing prediction records:", missing_results)
print("Extra prediction records:", extra_results)

validation_pass = (
    matched == len(dataset_records)
    and mismatched == 0
    and missing_results == 0
    and extra_results == 0
    and duplicate_original == 0
    and duplicate_results == 0
)

print("\n" + "=" * 60)

if validation_pass:
    print("OVERALL STATUS: PASS")
else:
    print("OVERALL STATUS: FAIL")

print("=" * 60)

RECORD MATCHING VALIDATION
Original records: 3448
Prediction records: 3448

Duplicate original keys: 0
Duplicate prediction keys: 0

Matched records: 3448
Mismatched records: 0
Missing prediction records: 0
Extra prediction records: 0

OVERALL STATUS: PASS


In [5]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(
    r"C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS"
)

QWEN_V2_RESULTS_FILE = (
    BASE_DIR
    / "results"
    / "qwen"
    / "prompt_v2"
    / "full_results_qwen_v2.jsonl"
)

qwen_results = []

with open(QWEN_V2_RESULTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            qwen_results.append(json.loads(line))

qwen_df = pd.DataFrame(qwen_results)

print("Total records:", len(qwen_df))
print("Successful:", (qwen_df["status"] == "success").sum())
print("Failed:", (qwen_df["status"] == "failed").sum())

print("\nDomain distribution:")
print(qwen_df["domain"].value_counts())

print("\nEmpty predictions:")
print(
    qwen_df["prediction"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

assert len(qwen_df) == 3448
assert qwen_df["record_id"].is_unique
assert (qwen_df["status"] == "success").all()

print("\nQWEN V2 RESULTS VALIDATION PASSED.")

Total records: 3448
Successful: 3448
Failed: 0

Domain distribution:
domain
Sentimen News    980
Penukaran Kod    799
Terjemahan       712
Sentimen         682
Campuran Kod     275
Name: count, dtype: int64

Empty predictions:
33

QWEN V2 RESULTS VALIDATION PASSED.


In [19]:
### FULL EVALUATION FOR QWEN 3.8 ###

In [20]:
%pip install -U sacrebleu bert-score scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
import re
import time
import pandas as pd
import numpy as np

from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
from sacrebleu.metrics import BLEU, CHRF
from bert_score import score as bert_score

# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path.cwd()

RESULT_FILE = (
    BASE_DIR / "results" / "qwen" / "prompt_v2" /"full_results_qwen_v2.jsonl"
)

OUTPUT_DIR = BASE_DIR / "results" / "qwen"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVALUATION_FILE = OUTPUT_DIR / "evaluation_results_qwen.csv"
DETAIL_FILE = OUTPUT_DIR / "evaluation_details_qwen.csv"
INVALID_FILE = OUTPUT_DIR / "invalid_predictions_qwen.csv"

# ============================================================
# 2. LOAD RESULTS
# ============================================================

with open(RESULT_FILE, "r", encoding="utf-8") as f:
    eval_records = [json.loads(line) for line in f if line.strip()]

print("=" * 60)
print("QWEN FULL EVALUATION")
print("=" * 60)
print("Total records:", len(eval_records))

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def normalize_key(value):
    return re.sub(r"\s+", " ", str(value)).strip().lower()


def get_field(data, aliases):
    """Get a field using normalized exact key matching."""
    if not isinstance(data, dict):
        return None

    normalized = {
        normalize_key(k): v
        for k, v in data.items()
    }

    for alias in aliases:
        key = normalize_key(alias)
        if key in normalized:
            return normalized[key]

    return None


def flatten_input(value):
    if isinstance(value, dict):
        return "\n".join(
            f"{k}: {v}"
            for k, v in value.items()
            if v is not None
        )
    return str(value or "")


def clean_text(value):
    if value is None:
        return ""

    text = str(value).strip()
    text = re.sub(r"^\s*(jawapan|answer|terjemahan)\s*:\s*", "", text, flags=re.I)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_text(value):
    text = clean_text(value).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def extract_sections(prediction):
    """
    Extract FORMAL and TIDAK FORMAL sections.
    Headers must appear at the beginning of a line.
    """
    text = str(prediction or "")

    pattern = re.compile(
        r"(?im)^\s*(?:[-*#]\s*)?"
        r"(TIDAK\s+FORMAL|FORMAL)\s*:\s*"
    )

    matches = list(pattern.finditer(text))
    sections = {}

    for i, match in enumerate(matches):
        label = re.sub(r"\s+", " ", match.group(1)).upper()

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        sections[label] = clean_text(text[start:end])

    return sections


def extract_options(input_data):
    """
    Extract A/B/C options from the original input.
    Supports formats such as:
    A. Positif
    B) Neutral
    C: Negatif
    """
    text = flatten_input(input_data)

    options = {}

    pattern = re.compile(
        r"(?im)^\s*([ABC])\s*[\.\):\-]\s*(.+?)\s*$"
    )

    for match in pattern.finditer(text):
        options[match.group(1).upper()] = clean_text(match.group(2))

    return options



def normalize_sentiment(value, input_data=None):
    if value is None:
        return None

    text = clean_text(value).strip().lower()

    # Remove option prefix: A. positif, B) neutral
    text = re.sub(r"^[abc]\s*[\.\):\-]\s*", "", text)

    aliases = {
        "positif": "Positif",
        "positive": "Positif",
        "neutral": "Neutral",
        "netural": "Neutral",
        "netutral": "Neutral",
        "netral": "Neutral",
        "negatif": "Negatif",
        "negative": "Negatif"
    }

    if text in aliases:
        return aliases[text]

    # Map A/B/C using the actual options in the input
    if text.upper() in {"A", "B", "C"} and input_data is not None:
        options = extract_options(input_data)
        option_value = options.get(text.upper())

        if option_value:
            option_key = clean_text(option_value).lower()

            if option_key in aliases:
                return aliases[option_key]

    return None


# ============================================================
# 4. PREPARE EVALUATION DATA
# ============================================================

text_tasks = {
    "Campuran Kod": [
        ("Formal", ["ujaran formal"]),
        ("Tidak Formal", ["ujaran tidak formal"])
    ],
    "Penukaran Kod": [
        ("Formal", ["jawapan formal"]),
        ("Tidak Formal", ["jawapan tidak formal"])
    ],
    "Terjemahan": [
        ("Terjemahan", ["jawapan (bahasa sasaran)"])
    ]
}

classification_tasks = {
    "Sentimen News": {
        "reference_aliases": ["sifat sentimen"],
        "input_aliases": ["teks"]
    },
    "Sentimen": {
        "reference_aliases": ["jawapan"],
        "input_aliases": ["soalan"]
    }
}

detail_rows = []
invalid_rows = []

for record in eval_records:

    domain = record["domain"]
    prediction = str(record.get("prediction") or "")
    reference_data = record.get("reference") or {}
    input_data = record.get("input") or {}

    # Text generation domains
    if domain in text_tasks:

        if domain in ["Campuran Kod", "Penukaran Kod"]:
            sections = extract_sections(prediction)

            for task_name, aliases in text_tasks[domain]:
                reference = get_field(reference_data, aliases)
                reference = clean_text(reference)

                section_key = task_name.upper()
                candidate = sections.get(section_key, "")

                if reference:
                    valid_prediction = bool(candidate.strip())

                    detail_rows.append({
                        "record_id": record["record_id"],
                        "domain": domain,
                        "task": task_name,
                        "reference": reference,
                        "prediction": candidate,
                        "valid_prediction": valid_prediction
                    })

                    if not valid_prediction:
                        invalid_rows.append({
                            "record_id": record["record_id"],
                            "domain": domain,
                            "task": task_name,
                            "reason": "Missing or unparsed output section"
                        })

        else:
            reference = get_field(
                reference_data,
                ["jawapan (bahasa sasaran)"]
            )

            reference = clean_text(reference)
            candidate = clean_text(prediction)

            if reference:
                valid_prediction = bool(candidate.strip())

                detail_rows.append({
                    "record_id": record["record_id"],
                    "domain": domain,
                    "task": "Terjemahan",
                    "reference": reference,
                    "prediction": candidate,
                    "valid_prediction": valid_prediction
                })

                if not valid_prediction:
                    invalid_rows.append({
                        "record_id": record["record_id"],
                        "domain": domain,
                        "task": "Terjemahan",
                        "reason": "Empty prediction"
                    })

    # Classification domains
    elif domain in classification_tasks:

        config = classification_tasks[domain]

        reference = get_field(
            reference_data,
            config["reference_aliases"]
        )

        reference_label = normalize_sentiment(
            reference,
            input_data
        )

        prediction_label = normalize_sentiment(
            prediction,
            input_data
        )

        if reference_label is not None:
            valid_prediction = prediction_label is not None

            detail_rows.append({
                "record_id": record["record_id"],
                "domain": domain,
                "task": "Classification",
                "reference": reference_label,
                "prediction": prediction_label or "INVALID",
                "valid_prediction": valid_prediction
            })

            if not valid_prediction:
                invalid_rows.append({
                    "record_id": record["record_id"],
                    "domain": domain,
                    "task": "Classification",
                    "reason": "Unrecognized sentiment label"
                })


details_df = pd.DataFrame(detail_rows)
invalid_df = pd.DataFrame(invalid_rows)

print("\nEvaluation items:", len(details_df))
print("Invalid predictions:", len(invalid_df))

# ============================================================
# 5. TEXT GENERATION METRICS
# ============================================================

bleu_metric = BLEU()
chrf_metric = CHRF(word_order=2)

evaluation_rows = []

text_df = details_df[
    details_df["domain"].isin(text_tasks.keys())
].copy()

for (domain, task), group in text_df.groupby(["domain", "task"]):

    references = group["reference"].fillna("").tolist()
    predictions = group["prediction"].fillna("").tolist()

    normalized_refs = [normalize_text(x) for x in references]
    normalized_preds = [normalize_text(x) for x in predictions]

    exact_matches = sum(
        p == r and p != ""
        for p, r in zip(normalized_preds, normalized_refs)
    )

    total = len(group)
    valid_count = int(group["valid_prediction"].sum())

    bleu_value = bleu_metric.corpus_score(
        predictions,
        [references]
    ).score

    chrf_value = chrf_metric.corpus_score(
        predictions,
        [references]
    ).score

    evaluation_rows.append({
        "domain": domain,
        "task": task,
        "total": total,
        "valid_predictions": valid_count,
        "invalid_predictions": total - valid_count,
        "valid_rate_percent": round(valid_count / total * 100, 2),
        "exact_match_percent": round(exact_matches / total * 100, 2),
        "BLEU": round(bleu_value, 4),
        "chrF++": round(chrf_value, 4),
        "BERTScore_P": np.nan,
        "BERTScore_R": np.nan,
        "BERTScore_F1": np.nan,
        "accuracy_percent": np.nan,
        "precision_macro_percent": np.nan,
        "recall_macro_percent": np.nan,
        "f1_macro_percent": np.nan,
        "f1_weighted_percent": np.nan
    })

# ============================================================
# 6. BERTSCORE
# ============================================================

print("\nCalculating BERTScore...")
print("Model: xlm-roberta-large")
print("Batch size: 16")

if len(text_df) > 0:

    bert_device = "cuda"

    try:
        import torch

        if not torch.cuda.is_available():
            bert_device = "cpu"

    except Exception:
        bert_device = "cpu"

    bert_results = {}

    for (domain, task), group in text_df.groupby(["domain", "task"]):

        references = group["reference"].fillna("").tolist()
        predictions = group["prediction"].fillna("").tolist()

        # Identify non-empty predictions
        valid_indices = [
            i for i, pred in enumerate(predictions)
            if str(pred).strip() != ""
        ]

        # Initialize all scores as zero
        P_scores = np.zeros(len(predictions))
        R_scores = np.zeros(len(predictions))
        F1_scores = np.zeros(len(predictions))

        # Calculate BERTScore only for non-empty predictions
        if valid_indices:

            valid_predictions = [
                predictions[i] for i in valid_indices
            ]

            valid_references = [
                references[i] for i in valid_indices
            ]

            P, R, F1 = bert_score(
                valid_predictions,
                valid_references,
                model_type="xlm-roberta-large",
                device=bert_device,
                batch_size=16,
                verbose=False
            )

            # Insert calculated scores into original positions
            for j, original_index in enumerate(valid_indices):
                P_scores[original_index] = P[j].item()
                R_scores[original_index] = R[j].item()
                F1_scores[original_index] = F1[j].item()

        bert_results[(domain, task)] = {
            "P": round(float(np.mean(P_scores)), 4),
            "R": round(float(np.mean(R_scores)), 4),
            "F1": round(float(np.mean(F1_scores)), 4)
        }

    # Insert BERTScore results into evaluation table
    for row in evaluation_rows:

        key = (row["domain"], row["task"])

        if key in bert_results:
            row["BERTScore_P"] = bert_results[key]["P"]
            row["BERTScore_R"] = bert_results[key]["R"]
            row["BERTScore_F1"] = bert_results[key]["F1"]

print("\nBERTScore calculation completed.")

# ============================================================
# 7. CLASSIFICATION METRICS
# ============================================================

classification_df = details_df[
    details_df["domain"].isin(classification_tasks.keys())
].copy()

CLASS_LABELS = ["Positif", "Neutral", "Negatif"]

for domain, group in classification_df.groupby("domain"):

    y_true = group["reference"].tolist()
    y_pred = group["prediction"].tolist()

    total = len(group)
    valid_count = int(group["valid_prediction"].sum())

    evaluation_rows.append({
        "domain": domain,
        "task": "Classification",
        "total": total,
        "valid_predictions": valid_count,
        "invalid_predictions": total - valid_count,
        "valid_rate_percent": round(valid_count / total * 100, 2),
        "exact_match_percent": np.nan,
        "BLEU": np.nan,
        "chrF++": np.nan,
        "BERTScore_P": np.nan,
        "BERTScore_R": np.nan,
        "BERTScore_F1": np.nan,
        "accuracy_percent": round(
            accuracy_score(y_true, y_pred) * 100, 2
        ),
        "precision_macro_percent": round(
            precision_score(
                y_true, y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0
            ) * 100, 2
        ),
        "recall_macro_percent": round(
            recall_score(
                y_true, y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0
            ) * 100, 2
        ),
        "f1_macro_percent": round(
            f1_score(
                y_true, y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0
            ) * 100, 2
        ),
        "f1_weighted_percent": round(
            f1_score(
                y_true, y_pred,
                labels=CLASS_LABELS,
                average="weighted",
                zero_division=0
            ) * 100, 2
        )
    })

# ============================================================
# 8. SAVE RESULTS
# ============================================================

evaluation_df = pd.DataFrame(evaluation_rows)

evaluation_df.to_csv(
    EVALUATION_FILE,
    index=False,
    encoding="utf-8-sig"
)

details_df.drop(
    columns=["reference", "prediction"]
).to_csv(
    DETAIL_FILE,
    index=False,
    encoding="utf-8-sig"
)

invalid_df.to_csv(
    INVALID_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("QWEN3.8 27B - EVALUATION SUMMARY")
print("=" * 70)

display(evaluation_df.round(4))

print("\nTotal evaluation items:", len(details_df))
print("Invalid predictions:", len(invalid_df))

print("\nFiles saved:")
print("Evaluation:", EVALUATION_FILE)
print("Details:", DETAIL_FILE)
print("Invalid:", INVALID_FILE)

print("\nEVALUATION COMPLETED")


QWEN FULL EVALUATION
Total records: 3448

Evaluation items: 4348
Invalid predictions: 45

Calculating BERTScore...
Model: xlm-roberta-large
Batch size: 16


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1897.71it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\AI\rocm-env\Lib\site-packages\transformers\integrations\sdpa_attention.py:174: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\transformers\hip\sdp_utils.cpp:310.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Loading weights: 100%|██████████| 391/391 [00


BERTScore calculation completed.

QWEN3.8 27B - EVALUATION SUMMARY


,domain,task,total,valid_predictions,invalid_predictions,valid_rate_percent,exact_match_percent,BLEU,chrF++,BERTScore_P,BERTScore_R,BERTScore_F1,accuracy_percent,precision_macro_percent,recall_macro_percent,f1_macro_percent,f1_weighted_percent
0,Campuran Kod,Formal,275,274,1,99.64,0.00,2.3008,27.1310,0.8713,0.8918,0.8812,NaN,NaN,NaN,NaN,NaN
1,Campuran Kod,Tidak Formal,101,101,0,100.00,0.00,2.6066,27.6539,0.8769,0.9139,0.8949,NaN,NaN,NaN,NaN,NaN
2,Penukaran Kod,Formal,799,796,3,99.62,0.00,3.0160,29.5512,0.8621,0.8862,0.8737,NaN,NaN,NaN,NaN,NaN
3,Penukaran Kod,Tidak Formal,799,788,11,98.62,0.00,1.1289,21.5336,0.8438,0.8674,0.8552,NaN,NaN,NaN,NaN,NaN
4,Terjemahan,Terjemahan,712,682,30,95.79,0.28,30.5275,54.2088,0.9023,0.9043,0.9032,NaN,NaN,NaN,NaN,NaN
5,Sentimen,Classification,682,682,0,100.00,NaN,NaN,NaN,NaN,NaN,NaN,76.98,72.67,72.37,72.51,76.82
6,Sentimen News,Classification,980,980,0,100.00,NaN,NaN,NaN,NaN,NaN,NaN,62.45,68.55,63.52,62.62,64.12



Total evaluation items: 4348
Invalid predictions: 45

Files saved:
Evaluation: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\results\qwen\evaluation_results_qwen.csv
Details: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\results\qwen\evaluation_details_qwen.csv
Invalid: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\results\qwen\invalid_predictions_qwen.csv

EVALUATION COMPLETED


In [7]:

print("=" * 65)
print("EVALUATION ITEM AUDIT")
print("=" * 65)

print("\nExpected vs Actual Evaluation Items:")

expected_counts = {
    "Campuran Kod": 550,
    "Penukaran Kod": 1598,
    "Terjemahan": 712,
    "Sentimen News": 980,
    "Sentimen": 682
}

actual_counts = details_df.groupby("domain").size().to_dict()

for domain, expected in expected_counts.items():
    actual = actual_counts.get(domain, 0)

    print(
        f"{domain:20} | "
        f"Expected: {expected:4} | "
        f"Actual: {actual:4} | "
        f"Difference: {actual - expected:+4}"
    )

print("\nTotal expected:", sum(expected_counts.values()))
print("Total actual:", len(details_df))

print("\nInvalid predictions by domain:")

if len(invalid_df):
    print(invalid_df.groupby("domain").size().to_string())
else:
    print("No invalid predictions.")

print("\nTask breakdown:")

print(
    details_df.groupby(["domain", "task"])
    .size()
    .to_string()
)

print("\nLibrary versions:")

import transformers
import bert_score
import torch

print("Transformers:", transformers.__version__)
print("BERTScore:", bert_score.__version__ if hasattr(bert_score, "__version__") else "Version unavailable")
print("PyTorch:", torch.__version__)
print("CUDA/ROCm available:", torch.cuda.is_available())


EVALUATION ITEM AUDIT

Expected vs Actual Evaluation Items:
Campuran Kod         | Expected:  550 | Actual:  376 | Difference: -174
Penukaran Kod        | Expected: 1598 | Actual: 1598 | Difference:   +0
Terjemahan           | Expected:  712 | Actual:  712 | Difference:   +0
Sentimen News        | Expected:  980 | Actual:  980 | Difference:   +0
Sentimen             | Expected:  682 | Actual:  682 | Difference:   +0

Total expected: 4522
Total actual: 4348

Invalid predictions by domain:
domain
Campuran Kod      1
Penukaran Kod    14
Terjemahan       30

Task breakdown:
domain         task          
Campuran Kod   Formal            275
               Tidak Formal      101
Penukaran Kod  Formal            799
               Tidak Formal      799
Sentimen       Classification    682
Sentimen News  Classification    980
Terjemahan     Terjemahan        712

Library versions:
Transformers: 5.22.1
BERTScore: 0.3.12
PyTorch: 2.9.1+rocm7.2.1
CUDA/ROCm available: True


In [8]:

from collections import Counter

sentiment_audit = Counter()
sentiment_issue_ids = []

for record in eval_records:
    domain = record.get("domain")

    if domain not in ["Sentimen News", "Sentimen"]:
        continue

    reference_data = record.get("reference") or {}
    input_data = record.get("input") or {}

    if domain == "Sentimen News":
        reference = get_field(reference_data, ["sifat sentimen"])
    else:
        reference = get_field(reference_data, ["jawapan"])

    if reference is None or not str(reference).strip():
        sentiment_audit["Missing reference field"] += 1
        sentiment_issue_ids.append(record["record_id"])
        continue

    normalized = normalize_sentiment(reference, input_data)

    if normalized is None:
        sentiment_audit["Reference label not recognized"] += 1
        sentiment_issue_ids.append(record["record_id"])
    else:
        sentiment_audit["Valid reference label"] += 1

print("=" * 60)
print("SENTIMENT REFERENCE AUDIT")
print("=" * 60)

for category, count in sentiment_audit.items():
    print(f"{category}: {count}")

print("\nTotal sentiment records:", sum(sentiment_audit.values()))
print("Records requiring inspection:", len(sentiment_issue_ids))


SENTIMENT REFERENCE AUDIT
Valid reference label: 1662

Total sentiment records: 1662
Records requiring inspection: 0


In [9]:

from collections import Counter

issue_summary = Counter()

for record in eval_records:
    domain = record.get("domain")

    if domain not in ["Sentimen News", "Sentimen"]:
        continue

    reference_data = record.get("reference") or {}
    input_data = record.get("input") or {}

    if domain == "Sentimen News":
        reference = get_field(reference_data, ["sifat sentimen"])
    else:
        reference = get_field(reference_data, ["jawapan"])

    if normalize_sentiment(reference, input_data) is not None:
        continue

    if reference is None:
        issue_summary[(domain, "Missing value")] += 1
        continue

    value = str(reference).strip()

    if not value:
        category = "Empty value"
    elif value.upper() in {"A", "B", "C"}:
        category = "Single option letter"
    elif len(value) <= 30:
        category = "Short unrecognized label"
    else:
        category = "Long or descriptive label"

    issue_summary[(domain, category)] += 1

print("REFERENCE ISSUE SUMMARY")
print("=" * 50)

for (domain, category), count in issue_summary.items():
    print(f"{domain} | {category}: {count}")

print("\nTotal problematic references:", sum(issue_summary.values()))


REFERENCE ISSUE SUMMARY

Total problematic references: 0


In [11]:

for record in eval_records:
    domain = record.get("domain")

    if domain not in ["Sentimen News", "Sentimen"]:
        continue

    reference_data = record.get("reference") or {}
    input_data = record.get("input") or {}

    if domain == "Sentimen News":
        reference = get_field(reference_data, ["sifat sentimen"])
    else:
        reference = get_field(reference_data, ["jawapan"])

    if normalize_sentiment(reference, input_data) is None:
        print("-" * 50)
        print("Domain:", domain)
        print("Record ID:", record["record_id"])
        print("Reference value:", repr(reference))
        print("Reference type:", type(reference).__name__)


In [ ]:
### COSINE SIMILARITY BY SENTENCE TRANSFORMERS (FOR SEMANTIC SIMILARITY ###

In [3]:
%pip install sentence-transformers

   ---------------------------------------- 0.0/780.3 kB ? eta -:--:--
   ---------------------------------------- 780.3/780.3 kB 11.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

semantic_model = SentenceTransformer(
    MODEL_NAME,
    device="cuda"
)

print("Model loaded successfully!")

C:\AI\rocm-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\WaRiSAN-01\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\AI\rocm-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `hugg

Model loaded successfully!


In [3]:
import torch
import transformers
import sentence_transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

print("CUDA available:", torch.cuda.is_available())
print(
    "MPS available:",
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
)

PyTorch: 2.9.1+rocm7.2.1
Transformers: 5.22.1
Sentence Transformers: 6.3.0
CUDA available: True
MPS available: False
